In [3]:
import csv
import numpy as np
from pathlib import Path
import sys, os
sys.path.append(os.path.join(os.getcwd(), '..'))
from utils.eda_helpers import read_csv_numeric

file_path = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
DATA_FOLDER = os.path.join(file_path, '..', 'dataset')

In [4]:
x_indices, x_data = read_csv_numeric(os.path.join(DATA_FOLDER, "x_train.csv"))
y_indices, y_data = read_csv_numeric(os.path.join(DATA_FOLDER, "y_train.csv"))

In [3]:
interesting_columns = ['_CASTHM1', 'FLUSHOT6', 'METVL11_', '_PASTRNG', 'EDUCA', 'CHCCOPD1', 'DRADVISE',
 'CHCKIDNY', 'ARTHSOCL', '_RFSEAT2', 'MENTHLTH', 'SMOKE100', 'DOCTDIAB', 'DIABAGE2', 'VETERAN3', 'CHKHEMO3', 'SEX', '_DRNKWEK', 'JOINPAIN', 'MARITAL', 'EXERANY2', '_PACAT1', 'INSULIN', 'DROCDY3_', 'DIFFALON',
 'FRUIT1', 'QLACTLM2', 'BLIND', 'RENTHOM1', 'PADUR2_', 'PAFREQ2_', 'LMTJOIN3', 'CHOLCHK', 'POORHLTH', 'DIABETE3',
 '_FRUTSUM', 'PERSDOC2', 'EXRACT21', 'FEETCHK', '_VEGESUM', 'PNEUVAC3', 'BPHIGH4', 'GENHLTH', 'EXRACT11',
 '_BMI5', 'DECIDE', '_ASTHMS1', '_MRACE1', '_LMTSCL1', 'ORNGDAY_', 'MSCODE', 'PHYSHLTH', 'TOLDHI2',
 'PADUR1_', 'DIFFWALK', 'BEANDAY_', 'CHCSCNCR', 'PAFREQ1_', '_AIDTST3', 'FLSHTMY2', 'PA1VIGM_',
 'METVL21_', 'SXORIENT', 'ARTHDIS2', 'PA1MIN_', 'MAXVO2_', '_RACE_G1', 'LONGWTCH', 'SCNTWRK1',
 'FEETCHK2', 'EMPLOY1', 'BLOODCHO', 'USENOW3', 'DRNKANY5', '_PAREC1', 'DIFFDRES', 'USEEQUIP', 'ADDEPEV2',
 '_AGE80', 'HTM4', 'INCOME2', 'CVDSTRK3', 'CHCOCNCR', 'MEDCOST', 'PREDIAB1', 'HAVARTH3', 'CHECKUP1',
 'TRNSGNDR', 'CHILDREN']

In [4]:
from utils.eda_helpers import clean_data
del x_indices["Id"]
x_data = clean_data(x_data, x_indices)

In [5]:
# --- Drop columns that are all NaN ---
valid_cols = ~np.isnan(x_data).all(axis=0)
x_data = x_data[:, valid_cols]
columns = [col for col, keep in zip(x_indices, valid_cols) if keep]
x_indices = {col: i for i, col in enumerate(columns)}

# --- Impute remaining NaNs with column mean ---
col_mean = np.nanmean(x_data, axis=0)
inds = np.where(np.isnan(x_data))
x_data[inds] = np.take(col_mean, inds[1])

# --- Standardize once ---
x_data = (x_data - np.mean(x_data, axis=0)) / np.std(x_data, axis=0)

# --- Prepare y ---
y = y_data[:,1].copy()
y[np.isnan(y)] = np.nanmean(y)
y = (y + 1) / 2  # convert -1,1 → 0,1
y = y.reshape(-1,1)

# --- Checks ---
print("Unique labels:", np.unique(y))
print("Any NaN in features?", np.any(np.isnan(x_data)))
print("Min std across features:", np.min(np.std(x_data, axis=0)))
print("Label distribution (mean):", np.mean(y))

C:\Users\clued\AppData\Local\Temp\ipykernel_5656\3791759394.py:13: RuntimeWarning: invalid value encountered in divide
  x_data = (x_data - np.mean(x_data, axis=0)) / np.std(x_data, axis=0)


Unique labels: [0. 1.]
Any NaN in features? True
Min std across features: nan
Any NaN in features? True
Min std across features: nan
Label distribution (mean): 0.08830207079403295


In [6]:
# --- Keep only the columns of interest ---
x_data_interested = x_data[:, [x_indices[col] for col in interesting_columns]]
x_indices_interested = {col: i for i, col in enumerate(interesting_columns)}

# --- Drop columns that are all NaN ---
valid_cols = ~np.isnan(x_data_interested).all(axis=0)
x_data_interested = x_data_interested[:, valid_cols]
interesting_columns = [col for col, keep in zip(interesting_columns, valid_cols) if keep]
x_indices_interested = {col: i for i, col in enumerate(interesting_columns)}

# --- Impute remaining NaNs with column mean ---
col_mean = np.nanmean(x_data_interested, axis=0)
inds = np.where(np.isnan(x_data_interested))
x_data_interested[inds] = np.take(col_mean, inds[1])

# --- Standardize once ---
x_data_interested = (x_data_interested - np.mean(x_data_interested, axis=0)) / np.std(x_data_interested, axis=0)

# --- Prepare y ---
y = y_data[:,1].copy()
y[np.isnan(y)] = np.nanmean(y)
y = (y + 1) / 2  # convert -1,1 → 0,1
y = y.reshape(-1,1)

# --- Checks ---
print("Unique labels:", np.unique(y))
print("Any NaN in features?", np.any(np.isnan(x_data_interested)))
print("Min std across features:", np.min(np.std(x_data_interested, axis=0)))
print("Label distribution (mean):", np.mean(y))

Unique labels: [0. 1.]
Any NaN in features? False
Min std across features: 0.9999999999999997
Label distribution (mean): 0.08830207079403295


In [7]:
from utils.all_functions import sigmoid
from implementations import logistic_regression, reg_logistic_regression

In [8]:
def weighted_logistic_loss(y, tx, w, sample_weights):
    pred = sigmoid(tx @ w)
    pred = np.clip(pred, 1e-15, 1-1e-15)
    loss = -np.sum(sample_weights * (y * np.log(pred) + (1 - y) * np.log(1 - pred))) / np.sum(sample_weights)
    return loss

def weighted_logistic_gradient(y, tx, w, sample_weights, lambda_=0):
    pred = sigmoid(tx @ w)
    grad = (tx.T @ (sample_weights * (pred - y))) / np.sum(sample_weights) + 2 * lambda_ * w
    return grad

In [9]:
def weighted_logistic_regression(y, tx, lambda_, initial_w, max_iters, gamma, tol=None):
    w = initial_w.copy()
    losses = []
    ws = []


    n_samples = len(y)
    n_pos = np.sum(y == 1)
    n_neg = n_samples - n_pos

    pos_weight = n_samples / (2*n_pos)
    neg_weight = n_samples / (2*n_neg)
    sample_weights = np.where(y == 1, pos_weight, neg_weight)

    for n_iter in range(max_iters):
        grad = weighted_logistic_gradient(y, tx, w, sample_weights, lambda_)
        w_new = w - gamma * grad
        loss = weighted_logistic_loss(y,tx,w,sample_weights)

        ws.append(w_new.copy())
        losses.append(loss)

        # Early stopping if tolerance specified
        if tol is not None and n_iter > 5:
            rel_change = abs(losses[-2] - losses[-1]) / (abs(losses[-2]) + 1e-10)
            if rel_change < tol:
                print(f"Stopping early at iter {n_iter} (Δloss_rel < tol)")
                break

        w = w_new

    return w, np.array(losses), ws


In [10]:
def run_experiment(y, tx, col_names, kind="logistic", 
                   gamma=0.01, lambda_=0.0, max_iters=1000, 
                   tol=None, w_init=None, verbose=True):
    """
    Run (regularized) logistic regression and visualize results.

    Args:
        y: np.ndarray, shape (N, 1)
        tx: np.ndarray, shape (N, D)
        col_names: list of feature names
        kind: "logistic" or "reg_logistic"
        gamma: learning rate
        lambda_: regularization strength
        max_iters: number of iterations
        tol: optional early stopping tolerance
        w_init: optional initial weight vector
        verbose: if True, prints info

    Returns:
        final_w, final_loss
    """
    n_features = tx.shape[1]
    if w_init is None:
        w_init = np.zeros((n_features, 1))

    if kind == "w_logistic":
        w, losses, ws = weighted_logistic_regression(y, tx, lambda_, w_init, max_iters, gamma, tol)
    elif kind == "logistic":
        w, losses, ws = logistic_regression(y, tx, w_init, max_iters, gamma, tol)
    elif kind == "reg_logistic":
        w, losses, ws = reg_logistic_regression(y, tx, lambda_, w_init, max_iters, gamma, tol)
    else:
        raise ValueError("kind must be 'logistic' or 'reg_logistic'")
    return w, losses[-1]

In [11]:
w, loss = run_experiment(y, x_data_interested, interesting_columns, 
                          kind="w_logistic", gamma=0.01, 
                            max_iters=300, lambda_=0, tol=1e-4)

Stopping early at iter 247 (Δloss_rel < tol)


In [12]:
w_tot, loss_tot = run_experiment(y, x_data, columns, 
                          kind="w_logistic", gamma=0.01, 
                            max_iters=300, lambda_=0, tol=1e-4)

In [13]:
def predict(tx, w, threshold=0.5, return_prob=False):
    prob = sigmoid(tx @ w)
    if return_prob:
        return prob
    return (prob >= threshold).astype(int)

In [ ]:
def compute_roc_auc(y, proba):
    # Sort by descending probability
    desc_idx = np.argsort(-proba)
    y_sorted = y[desc_idx]
    proba_sorted = proba[desc_idx]

    P = np.sum(y == 1)
    N = np.sum(y == 0)

    tpr = [0.0]
    fpr = [0.0]

    thresholds = np.unique(proba_sorted)
    for thr in thresholds:
        y_pred = (proba >= thr).astype(int)
        tp = np.sum((y_pred == 1) & (y == 1))
        fp = np.sum((y_pred == 1) & (y == 0))
        tpr.append(tp / (P + 1e-10))
        fpr.append(fp / (N + 1e-10))

    tpr.append(1.0)
    fpr.append(1.0)

    tpr = np.array(tpr)
    fpr = np.array(fpr)

    auc = np.trapz(tpr, fpr)
    return auc


In [15]:
def compute_training_error(y, tx, w, threshold=0.5, returning="error"):
    """
    Compute classification metrics for weighted logistic regression (NumPy-only).

    Parameters
    ----------
    y : np.ndarray
        True binary labels (0 or 1)
    tx : np.ndarray
        Feature matrix
    w : np.ndarray
        Trained weights
    threshold : float, optional
        Decision threshold for classification
    returning : str, optional
        Which metric to return. Options:
        "error", "accuracy", "precision", "recall", "f1", "roc_auc", or "all"

    Returns
    -------
    float or dict
        The requested metric value or a dict of all metrics.
    """
    # Predicted probabilities and binary labels
    proba = sigmoid(tx @ w)
    y_pred = (proba >= threshold).astype(int)

    # Confusion matrix components
    tp = np.sum((y_pred == 1) & (y == 1))
    tn = np.sum((y_pred == 0) & (y == 0))
    fp = np.sum((y_pred == 1) & (y == 0))
    fn = np.sum((y_pred == 0) & (y == 1))

    # Basic metrics
    accuracy = (tp + tn) / (tp + tn + fp + fn + 1e-10)
    error = 1 - accuracy
    precision = tp / (tp + fp + 1e-10)
    recall = tp / (tp + fn + 1e-10)
    f1 = 2 * precision * recall / (precision + recall + 1e-10)
    roc_auc = compute_roc_auc(y, proba)
    # Return
    metrics = {
        "error": error,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc
    }

    if returning == "all":
        return metrics
    elif returning in metrics:
        return metrics[returning]
    else:
        raise ValueError(f"Unknown metric '{returning}'")


In [16]:
def evaluate_thresholds(y, tx, w, thresholds=None):
    """
    Evaluate performance metrics (accuracy, error, precision, recall, F1, AUC)
    at multiple thresholds using compute_training_error().
    """
    if thresholds is None:
        thresholds = np.linspace(0.1, 0.9, 9)
    
    results = []
    
    for t in thresholds:
        metrics = compute_training_error(y, tx, w, threshold=t, returning="all")
        
        results.append({
            "threshold": t,
            "accuracy": metrics["accuracy"],
            "error": metrics["error"],
            "precision": metrics["precision"],
            "recall": metrics["recall"],
            "f1": metrics["f1"],
            "roc_auc": metrics["roc_auc"]
        })
    
    return results


In [17]:
results = evaluate_thresholds(y, x_data_interested, w)

print(f"{'Threshold':<10} {'Accuracy':<10} {'Error':<10} {'Prec':<10} {'Rec':<10} {'F1':<10} {'AUC':<10}")
for r in results:
    print(f"{r['threshold']:<10.2f} {r['accuracy']:<10.3f} {r['error']:<10.3f} "
          f"{r['precision']:<10.3f} {r['recall']:<10.3f} {r['f1']:<10.3f} {r['roc_auc']:<10.3f}")


AttributeError: module 'numpy' has no attribute 'trapezoid'

In [ ]:
results = evaluate_thresholds(y, x_data, w_tot)

print(f"{'Threshold':<10} {'Accuracy':<10} {'Error':<10} {'Prec':<10} {'Rec':<10} {'F1':<10} {'AUC':<10}")
for r in results:
    print(f"{r['threshold']:<10.2f} {r['accuracy']:<10.3f} {r['error']:<10.3f} "
          f"{r['precision']:<10.3f} {r['recall']:<10.3f} {r['f1']:<10.3f} {r['roc_auc']:<10.3f}")

Threshold  Accuracy   Error      Prec       Rec        F1         AUC       
0.10       0.912      0.088      0.000      0.000      0.000      0.500     
0.20       0.912      0.088      0.000      0.000      0.000      0.500     
0.30       0.912      0.088      0.000      0.000      0.000      0.500     
0.40       0.912      0.088      0.000      0.000      0.000      0.500     
0.50       0.912      0.088      0.000      0.000      0.000      0.500     
0.60       0.912      0.088      0.000      0.000      0.000      0.500     
0.70       0.912      0.088      0.000      0.000      0.000      0.500     
0.80       0.912      0.088      0.000      0.000      0.000      0.500     
0.90       0.912      0.088      0.000      0.000      0.000      0.500     
